# Linkages - Getting Started

Welcome to Linkages! This notebook will guide you through the basics of discovering connections between entities in alternative markets.

## What is Linkages?

Linkages is a RAG (Retrieval-Augmented Generation) and LLM-powered system that discovers and maps connections between:
- **Firms** - Investment managers and asset managers
- **Funds** - Investment vehicles
- **Deals** - Transactions and investments
- **Service Providers** - Legal, advisory, and support firms

## Learning Objectives

By the end of this notebook, you'll understand how to:
1. Load entity data
2. Create embeddings
3. Build a RAG pipeline
4. Discover connections
5. Analyze connection networks

## Setup

First, let's import the necessary modules and set up the environment.

In [ ]:
# Add parent directory to path
import sys
sys.path.insert(0, '../')

# Import linkages modules
from src.config import DEBUG, DATA_DIR
from src.data import DataLoader
from src.embeddings import EntityEmbedder
from src.rag import EntityIndexer, EntityRetriever
from src.llm import ConnectionReasoner
from src.graph import LinkageMapper

import json
from pathlib import Path

print(f"Debug mode: {DEBUG}")
print(f"Data directory: {DATA_DIR}")

## Step 1: Load Entity Data

Let's load the sample data for firms, funds, deals, and service providers.

In [ ]:
# Initialize data loader
loader = DataLoader(data_dir=Path('../data'))

# Load entities
firms = loader.load_csv('sample_firms.csv', 'firm')
funds = loader.load_csv('sample_funds.csv', 'fund')
deals = loader.load_csv('sample_deals.csv', 'deal')
providers = loader.load_csv('sample_providers.csv', 'service_provider')

print(f"Loaded {len(firms)} firms")
print(f"Loaded {len(funds)} funds")
print(f"Loaded {len(deals)} deals")
print(f"Loaded {len(providers)} service providers")

# Display sample entity
if firms:
    print(f"\nSample Firm:")
    print(f"  Name: {firms[0].name}")
    print(f"  AUM: ${firms[0].aum}M")
    print(f"  Description: {firms[0].description}")

## Step 2: Create Embeddings

Now let's generate embeddings for all entities using OpenAI's embedding model.

In [ ]:
# Initialize embedder
embedder = EntityEmbedder()

# Prepare all entities
all_entities = firms + funds + deals + providers

print(f"Total entities to embed: {len(all_entities)}")
print("Generating embeddings...")

# Generate embeddings
# Note: This will use OpenAI API, ensure you have OPENAI_API_KEY set
# embeddings = embedder.embed_entities(all_entities)
# print(f"Generated {len(embeddings)} embeddings")
# print(f"Embedding dimension: {len(embeddings[0])}")

print("✓ Embedder ready (embeddings not generated to save API calls in demo)")

## Step 3: Build RAG Pipeline

Let's create a vector store and index our entities.

In [ ]:
# Initialize indexer
indexer = EntityIndexer(persist_dir="../linkages_vectorstore")

print("Indexing entities...")
# This will create the vector store
# vectorstore = indexer.index_entities(all_entities)

print("✓ Indexer ready (index not created to save API calls in demo)")
print(f"Vector store persistence directory: ../linkages_vectorstore")

## Step 4: Discover Connections

Now let's use the linkage mapper to discover connections between entities.

In [ ]:
# Create entity dictionary
entities_dict = {e.id: e for e in all_entities}

print(f"Total entities in dictionary: {len(entities_dict)}")

# Display entity types
entity_types = {}
for e in all_entities:
    entity_types[e.entity_type] = entity_types.get(e.entity_type, 0) + 1

print("\nEntity types:")
for etype, count in entity_types.items():
    print(f"  {etype}: {count}")

In [ ]:
# Initialize components for linkage discovery
# retriever = EntityRetriever(vectorstore)
# reasoner = ConnectionReasoner()
# mapper = LinkageMapper(retriever, reasoner)

# print("Discovering connections...")
# connections = mapper.discover_full_network(entities_dict, use_llm_reasoning=False)
# print(f"Discovered {len(connections)} connections")

print("✓ Components ready (LLM connections not discovered to save API calls in demo)")
print(f"\nValid connection types:")
for conn_type, desc in LinkageMapper.VALID_CONNECTIONS.items():
    print(f"  {conn_type} -> {desc}")

## Step 5: Example Connections

Let's manually create some example connections to demonstrate the system.

In [ ]:
from src.data.models import Connection

# Create some example connections
example_connections = [
    Connection(
        source_id="firm_001",
        target_id="fund_001",
        source_type="firm",
        target_type="fund",
        relationship_type="firm_funds",
        strength=0.95,
        evidence="Blackstone manages BDT Infrastructure Partners fund"
    ),
    Connection(
        source_id="fund_001",
        target_id="deal_001",
        source_type="fund",
        target_type="deal",
        relationship_type="fund_deals",
        strength=0.90,
        evidence="Fund invests in TowerBrook Infrastructure deal"
    ),
    Connection(
        source_id="deal_001",
        target_id="provider_002",
        source_type="deal",
        target_type="service_provider",
        relationship_type="deal_providers",
        strength=0.85,
        evidence="Freshfields provides legal services for the deal"
    )
]

print(f"Created {len(example_connections)} example connections\n")

# Display connections
for i, conn in enumerate(example_connections, 1):
    source = entities_dict.get(conn.source_id)
    target = entities_dict.get(conn.target_id)
    print(f"Connection {i}:")
    print(f"  {source.name} ({source.entity_type}) -> {target.name} ({target.entity_type})")
    print(f"  Type: {conn.relationship_type}")
    print(f"  Strength: {conn.strength:.2f}")
    print(f"  Evidence: {conn.evidence}")
    print()

## Step 6: Analyze Connection Network

Let's analyze statistics about the discovered connections.

In [ ]:
# Create a simple linkage mapper to analyze our example connections
mapper = LinkageMapper(
    retriever=None,  # Not used for this analysis
    reasoner=None    # Not used for this analysis
)

# Get statistics
stats = mapper.get_connection_statistics(example_connections)

print("Network Statistics:")
print(f"  Total connections: {stats['total_connections']}")
print(f"  Entities connected: {stats['entities_connected']}")
print(f"  Average connection strength: {stats['average_strength']:.2f}")
print(f"\n  Connections by type:")
for conn_type, count in stats['connections_by_type'].items():
    print(f"    {conn_type}: {count}")

## Next Steps

Now that you've learned the basics, here's what you can do next:

1. **Set up the full RAG pipeline** - Enable API calls in this notebook
2. **Load your own data** - Use your own CSV files instead of samples
3. **Discover network paths** - Find how entities are connected through chains
4. **Visualize connections** - Create graphs of the connection network
5. **Analyze specific entities** - Deep dive into connections for specific firms or funds

Check out the other notebooks for more advanced examples!